## LAB: **ATTENTION IN COMPUTER VISION**

---

***STUDENT(S):*** 

---

In the context of machine learning, **attention** is a technique that mimics cognitive attention, defined as the ability to choose and concentrate on relevant stimuli. In other words, **attention is a method that tries to enhance the important parts while fading out the non-relevant information**. The idea of Attention Mechanisms was first popularly introduced in the domain of Natural Language Processing (NLP) in the NeurIPS 2017 paper by Google Brain, titled ["**Attention Is All You Need**"](https://proceedings.neurips.cc/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf).

In computer vision, attention has found its way into several applications, such as in the domains of image classification, image segmentation, and image captioning.

<figure>
<center><img src="https://miro.medium.com/max/1400/1*sRy3ukQziKP0TSQqlz3LCg.webp" width="80%"></center> 
<figcaption align = "center"> Attention maps in an image captioning problem. </figcaption>
</figure>

# **CBAM**: Convolutional Block Attention Module 

In this lab we are going to implement one of the most popular attention modules in computer vision: the **Convolutional Block Attention Module (CBAM)**. It contains two sequential sub-modules called the **Channel Attention Module (CAM)** and the **Spatial Attention Module (SAM)**, which are applied in that particular order. The authors of the [paper](https://openaccess.thecvf.com/content_ECCV_2018/papers/Sanghyun_Woo_Convolutional_Block_Attention_ECCV_2018_paper.pdf) point out that CBAM is applied at every convolutional block in deep networks to get subsequent "*Refined Feature Maps*" from the "*Input Intermediate Feature Maps*". 

Before going through the specifics and implementation of Spatial and Channel Attention respectively, let's get to know what these terms mean and why they are essential.

*   ***Spatial*** refers to the domain space encapsulated within each **feature map**. Spatial attention represents the attention mechanism/attention mask on the feature map, or a single cross-sectional slice of the tensor. Refining the feature maps using Spatial Attention, we are **enhancing the input to the subsequent convolutional layers** which thus improves the performance of the model.
*   ***Channel*** attention essentially provides a weight for each channel and thus **enhances those particular channels which are most contributing towards learning** and thus boosts the overall model performance.




# Dataset

In this lab we are going yo apply CBAM in a classification problem. For that, we have selected the [STL-10 dataset](https://cs.stanford.edu/~acoates/stl10/), including a white patch to some images (as displayed below). The task is to create a neural network that classifies both types of images.

<figure>
<center><img src="https://miro.medium.com/max/1018/1*rzmgXEF2VJbw6IAB2bA-Cw.webp" width="55%"></center> 
<figcaption align = "center"> STL images. Those images labeled as 1 belong to the class whose images have a white patch while those labeled as 0 are the ones that do not have the white patch.</figcaption>
</figure>

# Imports

In [ ]:
import os
from PIL import Image
import torchvision
from torchvision.datasets import STL10
from torchvision import datasets, transforms, utils
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np

import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'  #To get figures with high quality!

# Reproducibility
torch.manual_seed(0)
np.random.seed(0)

print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

# PART I: Loading and preprocessing STL-10 dataset

Download the STL-10 dataset using torchvision.



In [ ]:
# Original dataset
# NOTE: running locally, the data is stored on drive D: (not enough space on C:).
# Both splits share the same root so the 2.6 GB archive is only downloaded once.
DATA_ROOT = 'D:/dl_lab02/data/stl10'
data_train = STL10(DATA_ROOT, split='train', download=True)
data_test = STL10(DATA_ROOT, split='test', download=True)

Now we define a dataset class to define the corrupted version of the images and their labels. It takes as input the images from the original dataset.

In [ ]:
class STL10_patch(Dataset):

  def __init__(self, dataset_og):

    self.dataset_og = dataset_og

  def __len__(self):
    return len(self.dataset_og)
  
  def __getitem__(self, index):

    image = self.dataset_og.__getitem__(index)[0]

    # Split in image and labels
    stl_image, label = self.corrupt_patch(image)
    
    # Transform image
    stl_image = self.transform_img(Image.fromarray(stl_image))

    return stl_image, label

  def corrupt_patch(self, img):

    img = np.array(img)

    corrupt = np.random.randint(2)

    label = 0.

    if corrupt:
      # Set label to 1 and add the patch
      label = 1.
      pos_x = np.random.randint(96-16)
      pos_y = np.random.randint(96-16)
      img[pos_x:pos_x+16, pos_y:pos_y+16, :] = 1

    return img, torch.tensor(label)

  
  def transform_img(self, image):

    transform_ops = torchvision.transforms.Compose([
                                        torchvision.transforms.ToTensor(),
                                        torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    return transform_ops(image)


Let's show an example from our *corrupted* dataset. We define a method to visualize the images, as they are normalized.

In [ ]:
def img_show(img):
    img = img / 2 + 0.5     # unnormalize to plot
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

In [ ]:
# 'Corrupted' dataset
trainset = STL10_patch(data_train)
testset = STL10_patch(data_test)

print('Trainset: ', len(trainset))
print('Testset: ', len(testset))

In [ ]:
for i in range(len(trainset)):

  img, label_class = trainset.__getitem__(i)

  if label_class == 1:

    print('IMG shape: ', img.shape)
    img_show(img)

    break


One we have defined the Dataset, we can define train and test dataloaders!

In [ ]:
batch_size = 32
valid_size = 500

# Training partition
part_tr = torch.utils.data.random_split(trainset, [len(trainset)-valid_size, valid_size])[0]
# Use sampler for randomization
training_sampler = torch.utils.data.SubsetRandomSampler(range(len(part_tr)))
# Define the dataloader
trainloader = torch.utils.data.DataLoader(part_tr, batch_size=batch_size, sampler=training_sampler)

# Validation partition
part_val = torch.utils.data.random_split(trainset, [valid_size, len(trainset)-valid_size])[0]
# Use sampler for randomization
valid_sampler = torch.utils.data.SubsetRandomSampler(range(len(part_val)))
# Define the dataloader
validloader = torch.utils.data.DataLoader(part_val, batch_size=batch_size, sampler=valid_sampler)

# Test partition
# Use sampler for randomization
test_sampler = torch.utils.data.SubsetRandomSampler(range(len(testset)))
# Define the dataloader
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, sampler=test_sampler)

In [ ]:
print('TRAIN: ', len(part_tr))
print('VALIDATION: ', len(part_val))
print('TEST: ', len(testset))

# PART II: Define the CBAM

***SPATIAL ATTENTION MODULE (SAM)***

<figure>
<center><img src="https://blog.paperspace.com/content/images/2020/05/Capture-2.PNG" width="70%"></center> 
</figure>

Spatial Attention Module (SAM) is comprised of a three-fold sequential operation.  

*   **Channel Pool**: the Input Tensor of dimensions `(c × h × w)` is decomposed to 2 channels, i.e. `(2 × h × w)`, where each of the 2 channels represent Max Pooling and Average Pooling across the channels.
*   **Convolution layer** which outputs a 1-channel feature map, i.e., the dimension of the output is `(1 × h × w)`. Thus, this convolution layer is a spatial dimension preserving convolution and uses padding to do the same. In code, the convolution is followed by a Batch Norm layer to normalize and scale the output of convolution. However, the authors have also provided an option to use ReLU activation function after the Convolution layer, but by default it only uses Convolution + Batch Norm.
*   **Sigmoid Activation layer**, which will map all the values to a range between 0 and 1. 

This Spatial Attention mask is then applied to all the feature maps in the input tensor using a simple element-wise product.

> **Exercise**: complete the following code.

In [ ]:
# 1. Channel Pool
class ChannelPool(nn.Module):

    def forward(self, x):

        out = torch.cat( (torch.max(x,1)[0].unsqueeze(1), torch.mean(x,1).unsqueeze(1)), dim=1 )
        
        return out

In [ ]:
# 2. Convolution Layer (+ Batch Normalization + ReLU activation)

class BasicConv(nn.Module):

    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1, groups=1, relu=True, bn=True, bias=False):

        super(BasicConv, self).__init__()

        self.out_channels = out_planes
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_planes, eps=1e-5, momentum=0.01, affine=True) if bn else None
        self.relu = nn.ReLU() if relu else None

    def forward(self, x):

        x = self.conv(x)

        if self.bn is not None:
            x = self.bn(x)

        if self.relu is not None:
            x = self.relu(x)

        return x

In [ ]:
# === SPATIAL ATTENTION MODULE (SAM) === #

class SpatialGate(nn.Module):

    def __init__(self):

        super(SpatialGate, self).__init__()

        kernel_size = 7
        self.compress = ChannelPool()
        self.spatial = BasicConv(2, 1, kernel_size, stride=1, padding=(kernel_size-1) // 2, relu=False)

        # NOTE:
        # // -> Divides the number on its left by the number on its right, rounds down the answer, and returns a whole number.

    def forward(self, x):

        x_compress = self.compress(x)          # (B, 2, H, W): max and mean across channels
        x_out = self.spatial(x_compress)       # (B, 1, H, W): conv 7x7 + BN
        scale = torch.sigmoid(x_out)           # mask in [0, 1]

        return x * scale

***CHANNEL ATTENTION MODULE (CAM)***

<figure>
<center><img src="https://blog.paperspace.com/content/images/2020/05/Capture-3.PNG" width="70%"></center> 
</figure>

The Channel Attention Module (CAM) is another sequential operation but a bit more complex than Spatial Attention Module (SAM). 

*  It first decomposes the input tensor into 2 subsequent vectors of dimensionality `(c × 1 × 1)`. One of these vectors is generated by **Global Average Pooling (GAP)** while the other vector is generated by **Global Max Pooling (GMP)**. 

    *   **Average Pooling**: aggregates spatial information, it has a smoothing effect.
    *   **Max Pooling**: it preserves much richer contextual information in the form of edges of the object within the image which thus leads to finer channel attention. It preserves natural edges of the objects more precisely.

*  This vector is passed to a **Multi Layer Perceptron (MLP)** network which has a bottleneck whose width or number of neurons is decided by the reduction ratio (r). The higher the reduction ratio, the fewer the number of neurons in the bottleneck and vice versa.

*   The output is the passed to a **Sigmoid activation layer** which then maps the values within the range of 0 and 1.

> **Exercise**: complete the following code.


In [ ]:
# AUXILIAR MODULE

class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)

In [ ]:
# === CHANNEL ATTENTION MODULE (CAM) === #

class ChannelGate(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16):

        super(ChannelGate, self).__init__()

        self.gate_channels = gate_channels
        self.mlp = nn.Sequential(
            Flatten(),
            nn.Linear(gate_channels, gate_channels // reduction_ratio),
            nn.ReLU(),
            nn.Linear(gate_channels // reduction_ratio, gate_channels)
            )

    def forward(self, x):

        channel_att_sum = None

        avg_pool = F.avg_pool2d( x, (x.size(2), x.size(3)), stride=(x.size(2), x.size(3)))
        channel_att_avg = self.mlp(avg_pool)   # shared MLP

        max_pool = F.max_pool2d( x, (x.size(2), x.size(3)), stride=(x.size(2), x.size(3)))
        channel_att_max = self.mlp(max_pool)   # shared MLP

        channel_att = torch.sigmoid(channel_att_avg + channel_att_max)

        scale = channel_att.unsqueeze(2).unsqueeze(3).expand_as(x) # mask for the three channels

        return x * scale

***COMPLETE CBAM!***
> **Exercise**: complete the following code.

In [ ]:
# === CONVOLUTIONAL BLOCK ATTENTION MODULE (CBAM) === #

class CBAM(nn.Module):

    def __init__(self, gate_channels, reduction_ratio=16, no_spatial=False):
        super(CBAM, self).__init__()

        self.ChannelGate = ChannelGate(gate_channels, reduction_ratio)
        self.no_spatial=no_spatial

        if not no_spatial:
            self.SpatialGate = SpatialGate()

    def forward(self, x):

        x_out = self.ChannelGate(x)

        if not self.no_spatial:
            x_out = self.SpatialGate(x_out)

        return x_out

**CBAM** is applied as a layer in every convolutional block of a convolutional neural network model. It takes in a tensor containing the feature maps from the previous convolutional layer and first refines it by applying channel attention using **CAM**. Subsequently this refined tensor is passed to **SAM** where the spatial attention is applied, thus resulting in the output refined feature maps. 

Now let's define a simple network including CBAM.

> **Exercise**: complete the following code.

In [ ]:
# Convolutional block
class ConvPart(nn.Module):

    def __init__(self):

        super().__init__()

        # Convolution and Max Pooling
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=5, padding=2)

        # Batch Normalization
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(32)

        # Activation
        self.relu = nn.ReLU()

    def forward(self, x):

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.pool1(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.pool2(out)

        out = self.conv3(out)

        return out

In [ ]:
# Full network including CBAM
class NetCBAM(nn.Module):

    def __init__(self, gate_channels=32):

        super().__init__()

        self.gate_channels = gate_channels

        # NN Blocks
        self.conv = ConvPart()
        self.attn1 = CBAM(self.gate_channels)
        self.linear = nn.Linear(self.gate_channels, 1)

    def forward(self, x):

        out_conv = self.conv(x)
        out_attn = self.attn1(out_conv)
        out = out_attn.mean(3).mean(2)
        out = self.linear(out)
        out = torch.sigmoid(out)[:,0]

        return out, out_attn

The following class completes the implementation of our network, extending it with a training method.

> **Exercise**: complete the following code.

In [ ]:
class NetCBAM_extended(NetCBAM):

    def __init__(self, gate_channels=32, lr=1e-3,epochs=20):

        super().__init__(gate_channels)

        # TRAINING VARIABLES
        self.lr = lr
        self.optim = optim.Adam(self.parameters(), self.lr)
        self.epochs = epochs

        # CRITERION
        self.criterion = nn.BCELoss()   # the network already outputs probabilities (sigmoid)

        # LOSS EVOLUTION
        self.loss_during_training = []
        self.valid_loss_during_training = []

        # GPU
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)


    def trainloop(self, trainloader, validloader):

        for e in range(int(self.epochs)):

          self.train()

          running_loss = 0.

          for images, labels in trainloader:

            images = images.to(self.device)
            labels = labels.to(self.device)

            self.optim.zero_grad()

            pred, _ = self.forward(images)

            loss = self.criterion(pred, labels)

            loss.backward()

            self.optim.step()

            running_loss += loss.item()

          self.loss_during_training.append(running_loss/len(trainloader))

          # Validation
          # Turn off gradients for validation, saves memory and computations
          self.eval()   # BatchNorm in inference mode
          with torch.no_grad():

              running_loss = 0.

              for images,labels in validloader:

                  images = images.to(self.device)
                  labels = labels.to(self.device)

                  pred, _ = self.forward(images)

                  loss = self.criterion(pred,labels)

                  running_loss += loss.item()

              self.valid_loss_during_training.append(running_loss/len(validloader))

          print('\nTrain Epoch: {} -> Training Loss: {:.6f}'.format(e,self.loss_during_training[-1]))
          print('Train Epoch: {} -> Validation Loss: {:.6f}'.format(e,self.valid_loss_during_training[-1]))

    def eval_performance(self, dataloader):

        loss = 0
        accuracy = 0

        self.eval()
        # Turn off gradients for validation, saves memory and computations
        with torch.no_grad():

            for images,labels in dataloader:
                # Move input and label tensors to the default device
                images, labels = images.to(self.device), labels.to(self.device)
                probs, _ = self.forward(images)

                labels_pred = torch.round(probs)
                equals = (labels_pred == labels)
                accuracy += torch.mean(equals.type(torch.FloatTensor))

            return accuracy/len(dataloader)

> **Exercise**: Train the model for 30 epochs. Plot the evolution of train and validation losses. Is the network overfitting?

In [ ]:
n_epochs = 30

my_cbam = NetCBAM_extended(gate_channels=32, lr=1e-3, epochs=n_epochs)
my_cbam.trainloop(trainloader, validloader)

plt.figure(figsize=(8, 5))
plt.plot(my_cbam.loss_during_training, label='Training loss')
plt.plot(my_cbam.valid_loss_during_training, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('BCE loss')
plt.title('CBAM network - loss evolution')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

> **Exercise**: Evaluate the performance of the model in the different data partitions.

In [ ]:
acc_train_cbam = my_cbam.eval_performance(trainloader)
acc_valid_cbam = my_cbam.eval_performance(validloader)
acc_test_cbam = my_cbam.eval_performance(testloader)

print('CBAM network accuracy')
print('  Train:      {:.4f}'.format(acc_train_cbam))
print('  Validation: {:.4f}'.format(acc_valid_cbam))
print('  Test:       {:.4f}'.format(acc_test_cbam))

> **Exercise**: complete the following code to visualize some of the obtaines attention maps. Discuss the obtained results.

In [ ]:
img, labels = next(iter(testloader))
img, labels = img.to(my_cbam.device), labels.to(my_cbam.device)

my_cbam.eval()
with torch.no_grad():
    probs, attn = my_cbam.forward(img)


fig, axes = plt.subplots(10, 6, figsize=(5*5, 10*5))

for i in range(10):
    img_plot = np.transpose(img[i,:,:,:].cpu().detach().numpy(), (1, 2, 0))
    img_plot = img_plot / 2 + 0.5
    axes[i, 0].imshow(img_plot)
    axes[i, 0].set_title("Input Image (label={:d}, pred={:.2f})".format(int(labels[i].item()), probs[i].item()))
    axes[i, 1].imshow(attn[i,0,:,:].cpu().detach().numpy())
    axes[i, 1].set_title("Attention Mask Channel 0")
    axes[i, 2].imshow(attn[i,1,:,:].cpu().detach().numpy())
    axes[i, 2].set_title("Attention Mask Channel 1")
    axes[i, 3].imshow(attn[i,2,:,:].cpu().detach().numpy())
    axes[i, 3].set_title("Attention Mask Channel 2")
    axes[i, 4].imshow(attn[i,3,:,:].cpu().detach().numpy())
    axes[i, 4].set_title("Attention Mask Channel 3")
    axes[i, 5].imshow(attn[i,4,:,:].cpu().detach().numpy())
    axes[i, 5].set_title("Attention Mask Channel 4")

As a complement, we also plot the **spatial attention mask** produced by the SAM (the `scale` in `SpatialGate`), which is a single map shared by all the channels and is the clearest way to see *where* the network is looking.

In [ ]:
# Spatial attention mask (output of the sigmoid in the SAM) for the same images
with torch.no_grad():
    feat = my_cbam.conv(img)
    feat = my_cbam.attn1.ChannelGate(feat)
    sam = my_cbam.attn1.SpatialGate
    spatial_mask = torch.sigmoid(sam.spatial(sam.compress(feat)))   # (B, 1, 24, 24)

fig, axes = plt.subplots(2, 10, figsize=(25, 5.5))
for i in range(10):
    img_plot = np.transpose(img[i].cpu().numpy(), (1, 2, 0)) / 2 + 0.5
    axes[0, i].imshow(np.clip(img_plot, 0, 1))
    axes[0, i].set_title('label={:d}'.format(int(labels[i].item())))
    axes[0, i].axis('off')
    axes[1, i].imshow(spatial_mask[i, 0].cpu().numpy(), cmap='jet')
    axes[1, i].set_title('SAM mask')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

> **OPTIONAL EXERCISE**: compare the results obtained using the CBAM block with the ones obtained using a network that hasn't attention.

In [ ]:
# Same network WITHOUT the attention block (CBAM removed)
class NetNoAttn(nn.Module):

    def __init__(self, gate_channels=32):
        super().__init__()
        self.gate_channels = gate_channels
        self.conv = ConvPart()
        self.linear = nn.Linear(self.gate_channels, 1)

    def forward(self, x):
        out_conv = self.conv(x)
        out = out_conv.mean(3).mean(2)
        out = self.linear(out)
        out = torch.sigmoid(out)[:,0]
        return out, out_conv


class NetNoAttn_extended(NetNoAttn):
    # Re-use exactly the same training / evaluation code as the CBAM network
    __init__ = NetCBAM_extended.__init__
    trainloop = NetCBAM_extended.trainloop
    eval_performance = NetCBAM_extended.eval_performance


torch.manual_seed(0)
my_noattn = NetNoAttn_extended(gate_channels=32, lr=1e-3, epochs=n_epochs)
my_noattn.trainloop(trainloader, validloader)

In [ ]:
acc_train_na = my_noattn.eval_performance(trainloader)
acc_valid_na = my_noattn.eval_performance(validloader)
acc_test_na = my_noattn.eval_performance(testloader)

print('{:<12s} {:>10s} {:>12s}'.format('Partition', 'CBAM', 'No attention'))
for name, a, b in [('Train', acc_train_cbam, acc_train_na),
                   ('Validation', acc_valid_cbam, acc_valid_na),
                   ('Test', acc_test_cbam, acc_test_na)]:
    print('{:<12s} {:>10.4f} {:>12.4f}'.format(name, a, b))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
axes[0].plot(my_cbam.loss_during_training, label='CBAM')
axes[0].plot(my_noattn.loss_during_training, label='No attention')
axes[0].set_title('Training loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(my_cbam.valid_loss_during_training, label='CBAM')
axes[1].plot(my_noattn.valid_loss_during_training, label='No attention')
axes[1].set_title('Validation loss'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.show()